# 第22章　モデルのライフサイクル管理 ― 版を、確実に扱う

**『医療診断支援AI開発　社会実装編 ― 臨床現場に届ける（社会実装編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-social

## 更新で古い実力を失わない ― 破滅的忘却への備え

In [ ]:
loss = task_loss(model(x), y)
for name, p in model.named_parameters():
    if name not in fisher:                  # 新しく足した層などは罰しない
        continue
    # 総和は「重要度を掛けたあと」に取る。.sum() を pow(2) にだけ掛けると、
    # 先にスカラーへ潰れたところへ fisher（重みと同形のテンソル）が掛かり、
    # loss がスカラーでなくなって backward の前に落ちる。
    loss = loss + 0.5 * lam * (fisher[name] * (p - theta_star[name]).pow(2)).sum()
# fisher[name]: 旧データでの勾配二乗の平均（重要度）, theta_star: 旧版の重み（detach済みの定数）

## 次に何を学ばせるか ― 能動学習の獲得関数

In [ ]:
probs = predict_unlabeled(model, pool)          # 無ラベル集合の予測
score = entropy(probs)                           # 迷いの大きさ
score = deduplicate_by_diversity(score, feats)   # 似た症例の重複を抑える
to_label = pool[score.argsort()[-budget:]]       # 予算内で最も学びの多い順に